# Mumbai & Navi Mumbai AQI — ML Pipeline

This notebook implements the project pipeline for the Data Visualization rubric:

**Raw Kaggle CSV → audit → cleaning → feature engineering → Random Forest Regression → evaluation → Power BI-ready CSV**

The model predicts **AQI** from available pollutant, environmental, and time-based features.

### Outputs
- `aqi_outputs/mumbai_navi_mumbai_aqi_cleaned.csv`
- `aqi_outputs/mumbai_navi_mumbai_aqi_powerbi.csv`
- `aqi_outputs/model_metrics.csv`
- `aqi_outputs/feature_importance.csv`

> Put `enriched_aqi_dataset.csv` in the same folder as this notebook before running it.


In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn openpyxl

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

CSV_FILE = "enriched_aqi_dataset.csv"
OUTPUT_DIR = "aqi_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Input file:", CSV_FILE)
print("Output directory:", OUTPUT_DIR)


In [ ]:
# ============================================================
# LOAD RAW DATA
# ============================================================

if not os.path.exists(CSV_FILE):
    raise FileNotFoundError(
        f"Could not find '{CSV_FILE}'. "
        "Place the downloaded Kaggle CSV in the same folder as this notebook."
    )

df_raw = pd.read_csv(CSV_FILE)

print("Dataset loaded successfully.")
print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])
display(df_raw.head())


In [ ]:
# ============================================================
# INITIAL DATA AUDIT
# ============================================================

print("Shape:", df_raw.shape)

print("\nColumn names:")
print(df_raw.columns.tolist())

print("\nData types:")
display(df_raw.dtypes.to_frame("Data Type"))

print("\nMissing values:")
missing = df_raw.isnull().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("Missing Count"))

print("\nDuplicate rows:")
print(df_raw.duplicated().sum())


In [ ]:
# ============================================================
# STANDARDIZE COLUMN NAMES
# ============================================================

df = df_raw.copy()

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("%", "pct")
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
)

print(df.columns.tolist())


In [ ]:
# ============================================================
# COLUMN DETECTION
# ============================================================

def find_column(possible_names, columns):
    for name in possible_names:
        if name in columns:
            return name
    return None

city_col = find_column(["city"], df.columns)
station_col = find_column(["station", "station_name"], df.columns)
date_col = find_column(["last_update", "date", "datetime", "timestamp"], df.columns)
aqi_col = find_column(["aqi", "air_quality_index"], df.columns)
lat_col = find_column(["latitude", "lat"], df.columns)
lon_col = find_column(["longitude", "lon", "lng"], df.columns)

print("Detected columns:")
print("City:", city_col)
print("Station:", station_col)
print("Date:", date_col)
print("AQI:", aqi_col)
print("Latitude:", lat_col)
print("Longitude:", lon_col)


In [ ]:
# ============================================================
# VERIFY MUMBAI / NAVI MUMBAI
# ============================================================

if city_col is None:
    raise ValueError("City column could not be identified.")

print("Unique cities containing 'Mumbai':")
mumbai_cities = sorted(
    df.loc[
        df[city_col].astype(str).str.contains("Mumbai", case=False, na=False),
        city_col
    ].dropna().astype(str).unique()
)
print(mumbai_cities)

print("\nExact/case-insensitive Mumbai records:")
print(
    df[city_col].astype(str).str.fullmatch("Mumbai", case=False, na=False).sum()
)

print("\nExact/case-insensitive Navi Mumbai records:")
print(
    df[city_col].astype(str).str.fullmatch("Navi Mumbai", case=False, na=False).sum()
)

if not df[city_col].astype(str).str.fullmatch("Navi Mumbai", case=False, na=False).any():
    print("\nWARNING: Navi Mumbai was not found as an exact city value.")
    print("Do NOT fabricate Navi Mumbai records. Inspect the city names above before continuing.")


In [ ]:
# ============================================================
# FILTER PROJECT CITIES
# ============================================================

df[city_col] = df[city_col].astype(str).str.strip()

city_mask = df[city_col].str.fullmatch(
    r"(?i)(Mumbai|Navi Mumbai)",
    na=False
)

df_project = df[city_mask].copy()

print("Project dataset shape:", df_project.shape)
print("\nCity distribution:")
display(df_project[city_col].value_counts())


In [ ]:
# ============================================================
# NUMERIC CONVERSION
# ============================================================

numeric_candidates = [
    aqi_col,
    lat_col,
    lon_col,
    "pollutant_min",
    "pollutant_max",
    "pollutant_avg",
    "temperature_c",
    "humidity_pct",
    "wind_speed_kmh"
]

numeric_candidates = [
    col for col in numeric_candidates
    if col is not None and col in df_project.columns
]

for col in numeric_candidates:
    df_project[col] = pd.to_numeric(df_project[col], errors="coerce")

print("Converted numeric columns:")
print(numeric_candidates)


In [ ]:
# ============================================================
# DATE/TIME PROCESSING
# ============================================================

if date_col:
    df_project[date_col] = pd.to_datetime(df_project[date_col], errors="coerce")

    print("Date range:")
    print(df_project[date_col].min(), "to", df_project[date_col].max())
else:
    print("WARNING: Date column not found.")


In [ ]:
# ============================================================
# REMOVE EXACT DUPLICATES
# ============================================================

before = len(df_project)
df_project = df_project.drop_duplicates().copy()
after = len(df_project)

print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before - after)


In [ ]:
# ============================================================
# TIME FEATURE ENGINEERING
# ============================================================

if date_col:
    df_project["date"] = df_project[date_col].dt.date
    df_project["year"] = df_project[date_col].dt.year
    df_project["month"] = df_project[date_col].dt.month
    df_project["month_name"] = df_project[date_col].dt.month_name()
    df_project["quarter"] = df_project[date_col].dt.quarter
    df_project["day"] = df_project[date_col].dt.day
    df_project["day_of_week"] = df_project[date_col].dt.dayofweek
    df_project["day_name"] = df_project[date_col].dt.day_name()
    df_project["hour"] = df_project[date_col].dt.hour
    df_project["weekend"] = (df_project[date_col].dt.dayofweek >= 5).astype(int)

    display(df_project[
        ["date", "year", "month", "month_name", "quarter",
         "day_of_week", "day_name", "hour", "weekend"]
    ].head())


In [ ]:
# ============================================================
# SEASON
# ============================================================

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Summer"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    elif month in [10, 11]:
        return "Post-Monsoon"
    return np.nan

df_project["season"] = df_project["month"].apply(get_season)

display(df_project["season"].value_counts())


In [ ]:
# ============================================================
# AQI CATEGORY
# ============================================================

if aqi_col is None:
    raise ValueError("AQI column was not found.")

def aqi_category(aqi):
    if pd.isna(aqi):
        return np.nan
    if aqi <= 50:
        return "Good"
    elif aqi <= 100:
        return "Satisfactory"
    elif aqi <= 200:
        return "Moderate"
    elif aqi <= 300:
        return "Poor"
    elif aqi <= 400:
        return "Very Poor"
    else:
        return "Severe"

df_project["aqi_category"] = df_project[aqi_col].apply(aqi_category)

display(df_project["aqi_category"].value_counts())


In [ ]:
# ============================================================
# SAVE CLEANED DATA BEFORE ML
# ============================================================

cleaned_file = os.path.join(
    OUTPUT_DIR,
    "mumbai_navi_mumbai_aqi_cleaned.csv"
)

df_project.to_csv(cleaned_file, index=False)

print("Saved:", cleaned_file)


In [ ]:
# ============================================================
# BASIC EXPLORATION
# ============================================================

summary = (
    df_project
    .groupby(city_col)[aqi_col]
    .agg(
        Count="count",
        Mean_AQI="mean",
        Median_AQI="median",
        Minimum_AQI="min",
        Maximum_AQI="max"
    )
    .round(2)
)

display(summary)


In [ ]:
# ============================================================
# AQI DISTRIBUTION
# ============================================================

plt.figure(figsize=(8, 5))
sns.boxplot(data=df_project, x=city_col, y=aqi_col)
plt.title("AQI Distribution: Mumbai vs Navi Mumbai")
plt.xlabel("City")
plt.ylabel("AQI")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# DETECT LONG-FORM POLLUTANT DATA
# ============================================================

long_format = (
    "pollutant_id" in df_project.columns
    and "pollutant_avg" in df_project.columns
)

print("Long-format pollutant data detected:", long_format)


In [ ]:
# ============================================================
# PIVOT LONG-FORM POLLUTANTS IF NECESSARY
# ============================================================

if long_format:
    group_columns = []

    for col in [
        city_col, station_col, date_col, lat_col, lon_col,
        "temperature_c", "humidity_pct", "wind_speed_kmh",
        aqi_col, "year", "month", "day", "hour",
        "day_of_week", "weekend", "season", "aqi_category"
    ]:
        if col and col in df_project.columns:
            group_columns.append(col)

    df_ml = (
        df_project
        .pivot_table(
            index=group_columns,
            columns="pollutant_id",
            values="pollutant_avg",
            aggfunc="mean"
        )
        .reset_index()
    )

    df_ml.columns.name = None
    df_ml.columns = [
        str(col).lower().strip().replace(".", "_").replace(" ", "_")
        for col in df_ml.columns
    ]

    print("Pivoted ML dataset shape:", df_ml.shape)
    display(df_ml.head())
else:
    df_ml = df_project.copy()
    print("Dataset already appears to be in wide format.")


In [ ]:
# ============================================================
# FINAL ML FEATURE DETECTION
# ============================================================

exclude_columns = {
    aqi_col, date_col, "date", "month_name", "day_name",
    "aqi_category", "predicted_aqi"
}
exclude_columns = {x for x in exclude_columns if x is not None}

numeric_columns = df_ml.select_dtypes(include=np.number).columns.tolist()

feature_columns = [
    col for col in numeric_columns
    if col not in exclude_columns
]

# Do not use coordinates as predictors by default.
feature_columns = [
    col for col in feature_columns
    if col not in [lat_col, lon_col]
]

print("Final model features:")
for feature in feature_columns:
    print(" -", feature)

print("\nTarget:", aqi_col)


In [ ]:
# ============================================================
# REMOVE ROWS WITHOUT TARGET AQI
# ============================================================

df_ml = df_ml.dropna(subset=[aqi_col]).copy()

print("Rows available for modelling:", len(df_ml))


In [ ]:
# ============================================================
# CHRONOLOGICAL TRAIN / TEST SPLIT
# ============================================================

if date_col in df_ml.columns:
    df_ml = df_ml.sort_values(by=date_col).reset_index(drop=True)
else:
    print("WARNING: Date column unavailable. Using existing row order.")

split_index = int(len(df_ml) * 0.80)

train_df = df_ml.iloc[:split_index].copy()
test_df = df_ml.iloc[split_index:].copy()

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

if date_col in df_ml.columns:
    print("\nTraining period:", train_df[date_col].min(), "to", train_df[date_col].max())
    print("Testing period:", test_df[date_col].min(), "to", test_df[date_col].max())


In [ ]:
# ============================================================
# X / y
# ============================================================

X_train = train_df[feature_columns].copy()
y_train = train_df[aqi_col].copy()

X_test = test_df[feature_columns].copy()
y_test = test_df[aqi_col].copy()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


In [ ]:
# ============================================================
# MEDIAN IMPUTATION
# ============================================================

train_medians = X_train.median()

X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print("Remaining missing values in X_train:", X_train.isnull().sum().sum())
print("Remaining missing values in X_test:", X_test.isnull().sum().sum())


In [ ]:
# ============================================================
# RANDOM FOREST REGRESSION
# ============================================================

model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Random Forest model trained successfully.")


In [ ]:
# ============================================================
# PREDICTIONS
# ============================================================

y_pred = model.predict(X_test)

print("First 10 predictions:")
print(y_pred[:10])


In [ ]:
# ============================================================
# MODEL EVALUATION
# ============================================================

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

metrics = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Value": [mae, rmse, r2]
})

display(metrics.style.format({"Value": "{:.4f}"}))


In [ ]:
# ============================================================
# CREATE ML OUTPUT COLUMNS
# ============================================================

test_output = test_df.copy()

test_output["predicted_aqi"] = y_pred
test_output["prediction_error"] = test_output[aqi_col] - test_output["predicted_aqi"]
test_output["absolute_error"] = test_output["prediction_error"].abs()
test_output["residual"] = test_output[aqi_col] - test_output["predicted_aqi"]
test_output["model_split"] = "Test"

display(test_output[
    [aqi_col, "predicted_aqi", "prediction_error", "absolute_error", "residual"]
].head())


In [ ]:
# ============================================================
# CREATE POWER BI DATASET
# ============================================================

powerbi_df = df_ml.copy()

powerbi_df["predicted_aqi"] = np.nan
powerbi_df["prediction_error"] = np.nan
powerbi_df["absolute_error"] = np.nan
powerbi_df["residual"] = np.nan
powerbi_df["model_split"] = "Train"

# Align using the chronological row index.
test_indices = test_df.index

powerbi_df.loc[test_indices, "predicted_aqi"] = y_pred
powerbi_df.loc[test_indices, "prediction_error"] = (
    test_df[aqi_col].to_numpy() - y_pred
)
powerbi_df.loc[test_indices, "absolute_error"] = np.abs(
    test_df[aqi_col].to_numpy() - y_pred
)
powerbi_df.loc[test_indices, "residual"] = (
    test_df[aqi_col].to_numpy() - y_pred
)
powerbi_df.loc[test_indices, "model_split"] = "Test"

powerbi_df["predicted_aqi_category"] = powerbi_df["predicted_aqi"].apply(aqi_category)

display(powerbi_df.head())


In [ ]:
# ============================================================
# FEATURE IMPORTANCE
# ============================================================

feature_importance = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

display(feature_importance)


In [ ]:
# ============================================================
# FEATURE IMPORTANCE CHART
# ============================================================

plt.figure(figsize=(10, 6))
sns.barplot(
    data=feature_importance,
    x="Importance",
    y="Feature"
)
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# ACTUAL VS PREDICTED AQI
# ============================================================

plt.figure(figsize=(8, 7))

plt.scatter(y_test, y_pred, alpha=0.5)

min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.xlabel("Actual AQI")
plt.ylabel("Predicted AQI")
plt.title("Actual vs Predicted AQI")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# PREDICTION ERROR DISTRIBUTION
# ============================================================

plt.figure(figsize=(9, 5))

sns.histplot(
    test_output["prediction_error"],
    bins=40,
    kde=True
)

plt.axvline(0, linestyle="--")

plt.title("Prediction Error Distribution")
plt.xlabel("Actual AQI - Predicted AQI")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# CITY-LEVEL MODEL PERFORMANCE
# ============================================================

if city_col in test_output.columns:
    city_metrics = []

    for city, group in test_output.groupby(city_col):
        city_mae = mean_absolute_error(group[aqi_col], group["predicted_aqi"])
        city_rmse = np.sqrt(mean_squared_error(group[aqi_col], group["predicted_aqi"]))
        city_r2 = r2_score(group[aqi_col], group["predicted_aqi"])

        city_metrics.append({
            "City": city,
            "MAE": city_mae,
            "RMSE": city_rmse,
            "R2": city_r2
        })

    city_metrics = pd.DataFrame(city_metrics)

    display(city_metrics.style.format({
        "MAE": "{:.2f}",
        "RMSE": "{:.2f}",
        "R2": "{:.3f}"
    }))
else:
    city_metrics = pd.DataFrame()


In [ ]:
# ============================================================
# SAVE ALL OUTPUTS
# ============================================================

cleaned_file = os.path.join(
    OUTPUT_DIR,
    "mumbai_navi_mumbai_aqi_cleaned.csv"
)

powerbi_file = os.path.join(
    OUTPUT_DIR,
    "mumbai_navi_mumbai_aqi_powerbi.csv"
)

metrics_file = os.path.join(
    OUTPUT_DIR,
    "model_metrics.csv"
)

importance_file = os.path.join(
    OUTPUT_DIR,
    "feature_importance.csv"
)

df_project.to_csv(cleaned_file, index=False)
powerbi_df.to_csv(powerbi_file, index=False)
metrics.to_csv(metrics_file, index=False)
feature_importance.to_csv(importance_file, index=False)

if not city_metrics.empty:
    city_metrics.to_csv(
        os.path.join(OUTPUT_DIR, "city_model_metrics.csv"),
        index=False
    )

print("Outputs created:")
for path in [
    cleaned_file,
    powerbi_file,
    metrics_file,
    importance_file
]:
    print(" -", path)

if not city_metrics.empty:
    print(" -", os.path.join(OUTPUT_DIR, "city_model_metrics.csv"))


In [ ]:
# ============================================================
# FINAL VALIDATION
# ============================================================

print("=" * 60)
print("FINAL PROJECT DATASET VALIDATION")
print("=" * 60)

print("\nShape:", powerbi_df.shape)

print("\nCities:")
if city_col in powerbi_df.columns:
    print(powerbi_df[city_col].value_counts())

print("\nAQI statistics:")
display(powerbi_df[aqi_col].describe())

print("\nModel metrics:")
display(metrics)

print("\nFinal Power BI columns:")
print(powerbi_df.columns.tolist())

print("\nOutput directory:")
print(os.path.abspath(OUTPUT_DIR))
